<a href="https://colab.research.google.com/github/Eden-Cho/Oral-Pill-Objest-Detection_TEAM_04/blob/seola/base_line_Faster_R_CNN%2BEfficientNet_B3_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> (중요) 여러분 구글 드라이브에 최소 7GB 이상은 확보되어 있어야 합니다!

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import unicodedata  # 0번 섹션에 추가 필요

# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

> (주의) 아래 코드는 처음 딱 한 번만!

In [ ]:
# import zipfile
# import os
# import shutil
# import time

# # 1. 경로 설정
# dataset_zip = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/초급_프로젝트_수강생_배포용.zip")
# extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/")

# os.makedirs(extract_path, exist_ok=True)

# # 2. 메인 압축파일 해제
# print(f"📦 메인 데이터셋 해제 중: {os.path.basename(dataset_zip)}")
# with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
#     zip_ref.extractall(extract_path)

# # 메인 압축파일 삭제 (요청사항)
# os.remove(dataset_zip)
# print("🗑️ 메인 압축파일 삭제 완료.")

# # 3. 내부 이미지 압축파일 통합 해제 로직
# print("\n🚀 이미지 폴더 통합 및 내부 압축 해제 시작...")
# for file in os.listdir(extract_path):
#     if file.endswith(".zip"):
#         file_path = os.path.join(extract_path, file)

#         # 이름 기반 대상 폴더 결정 (train/test 통합)
#         if 'train' in file.lower():
#             target_folder_name = "train_images"
#         elif 'test' in file.lower():
#             target_folder_name = "test_images"
#         else:
#             target_folder_name = file.replace(".zip", "")

#         target_subfolder = os.path.join(extract_path, target_folder_name)
#         os.makedirs(target_subfolder, exist_ok=True)

#         print(f"📂 {file} -> {target_folder_name} 통합 중...")

#         with zipfile.ZipFile(file_path, 'r') as zip_ref:
#             for member in zip_ref.infolist():
#                 if not member.is_dir():
#                     # 내부 경로 구조를 무시하고 파일명만 추출하여 저장
#                     filename = os.path.basename(member.filename)
#                     if filename:
#                         target_file_path = os.path.join(target_subfolder, filename)
#                         with zip_ref.open(member) as source, open(target_file_path, "wb") as target:
#                             shutil.copyfileobj(source, target)

#         # [수정 포인트] 삭제 전 잠시 대기 후 강제 삭제 시도
#         try:
#             time.sleep(0.5)
#             if os.path.exists(file_path):
#                 os.remove(file_path)
#                 print(f"🗑️ 삭제 성공: {file}")
#         except Exception as e:
#             print(f"❌ {file} 삭제 실패: {e}")

# print("\n✨ 모든 작업이 완료되었습니다!")
# print(f"📁 최종 데이터셋 구성: {os.listdir(extract_path)}")

- 구글 드라이브 휴지통 비우기

In [ ]:
# from google.colab import auth
# from googleapiclient.discovery import build

# # 1. 구글 드라이브 인증
# auth.authenticate_user()
# drive_service = build('drive', 'v3')

# # 2. 휴지통 완전히 비우기 함수
# def empty_trash():
#     try:
#         drive_service.files().emptyTrash().execute()
#         print("✅ 구글 드라이브 휴지통이 완전히 비워졌습니다.")
#     except Exception as e:
#         print(f"❌ 휴지통 비우기 실패: {e}")

# # 실행
# empty_trash()

> 압축 해제한 파일들의 반영 시간이 걸릴 수 있으므로, 커널 재시작 해주기!

#### `normalize_path`는?

`normalize_path`는 파일 경로에 포함된 **한글(유니코드) 처리 방식**을 통일하여, 경로를 찾지 못하는 에러를 방지하기 위한 함수입니다.

특히 **Google Colab**이나 **Mac, Windows** 사이에서 데이터를 주고받을 때 한글 폴더명이 깨져서 발생하는 `File Not Found` 에러를 잡는 데 필수적입니다.

<br>

##### 왜 사용하나요? (NFC vs NFD)

한글을 컴퓨터가 인식하는 방식은 크게 두 가지입니다.

* **NFC (Windows 스타일):** '강'을 '강'이라는 하나의 글자로 저장합니다.
* **NFD (Mac/Unix 스타일):** '강'을 'ㄱ', 'ㅏ', 'ㅇ'으로 쪼개서 저장합니다.

사람 눈에는 똑같이 "초급 프로젝트"라고 보이지만, 컴퓨터 입장에서는 글자 조합 방식이 다르면 **완전히 다른 경로**로 인식합니다. `normalize_path` 는 이를 **NFC(표준 방식)** 로 강제 통일해주는 역할을 합니다.

In [ ]:
!pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.7 MB/s eta 0:00:00


In [4]:
############################################################
# 0. 라이브러리 임포트 & 경로 설정
############################################################
import os
import json
import pandas as pd
from PIL import Image
import unicodedata
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from collections import OrderedDict

from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import FeaturePyramidNetwork, MultiScaleRoIAlign
from torchvision.ops.feature_pyramid_network import LastLevelMaxPool


# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

# 경로는 환경에 맞게 수정
# train_images, test_images
extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/") # 압축을 풀 폴더

TRAIN_JSON_PATH = os.path.join(extract_path, "merged_annotations_train_final.json")
TEST_JSON_PATH = os.path.join(extract_path, "merged_annotations_test_final.json")
TRAIN_IMG_DIR = os.path.join(extract_path, "train_images")
TEST_IMG_DIR  = os.path.join(extract_path, "test_images")

# merged_annotation json 경로
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [10]:

# 코랩 추가 데이터 파일 압축 해제

import zipfile

# 1. 업로드된 파일명 확인 (코랩 로컬 /content/ 기준)
# 혹시 파일명 뒤에 (1) 등이 붙었는지 확인하기 위해 리스트를 뽑습니다.
current_files = os.listdir('/content/')
print("현재 업로드된 파일 목록:", current_files)

# 2. 압축 해제 설정
zip_targets = {
    "images": "/content/new_images_1000.zip",
    "labels": "/content/TL_81_단일.zip"
}

extract_dirs = {
    "images": "/content/new_images_1000",
    "labels": "/content/TL_81_단일"
}

for key, zip_path in zip_targets.items():
    if os.path.exists(zip_path):
        target_dir = extract_dirs[key]
        if not os.path.exists(target_dir):
            print(f"📦 {zip_path} 압축 해제 중...")
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(target_dir)
            print(f"✅ {target_dir} 생성 완료!")
        else:
            print(f"📂 이미 압축이 풀려있습니다: {target_dir}")
    else:
        print(f"❌ 에러: {zip_path} 파일이 /content/ 안에 없습니다. 업로드를 완료해 주세요!")


# 4. 이후 코드에서 사용할 경로 변수 설정
EXTRA_IMG_DIR = "/content/new_images_1000"
EXTRA_JSON_DIR = "/content/TL_81_단일"

현재 업로드된 파일 목록: ['.config', 'drive', 'TL_81_단일.zip', 'new_images_1000', 'new_images_1000.zip', 'TL_81_단일', 'sample_data']
📂 이미 압축이 풀려있습니다: /content/new_images_1000
📂 이미 압축이 풀려있습니다: /content/TL_81_단일


In [7]:
############################################################
# 1. 원본 데이터 로드 + 추가 데이터(1000장) 병합
############################################################

def build_df_from_merged_json(json_path, img_dir):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    id_to_fname = {img["id"]: img["file_name"] for img in data["images"]}
    records = []
    for ann in data["annotations"]:
        img_id_coco = ann["image_id"]
        file_name = id_to_fname.get(img_id_coco)
        if file_name is None: continue
        img_path = os.path.join(img_dir, file_name)
        if not os.path.exists(img_path): continue
        x, y, w, h = ann["bbox"]
        records.append({
            "image_path": img_path,
            "image_id": os.path.splitext(file_name)[0],
            "category_id": int(ann["category_id"]),
            "bbox_x": float(x), "bbox_y": float(y), "bbox_w": float(w), "bbox_h": float(h),
        })
    return pd.DataFrame(records)

# 1-1. 기본 데이터 로드
df = build_df_from_merged_json(TRAIN_JSON_PATH, TRAIN_IMG_DIR)

# 1-2. 추가 이미지 및 라벨 병합 로직 (YOLO 코드에서 가져옴)
EXTRA_IMG_DIR = "/content/new_images_1000/new_images_1000"
EXTRA_JSON_DIR = "/content/TL_81_단일/TL_81_단일"

extra_records = []
existing_ids = set(df["image_id"].tolist())

if os.path.exists(EXTRA_IMG_DIR):
    for img_fname in os.listdir(EXTRA_IMG_DIR):
        if not img_fname.lower().endswith(('.jpg', '.jpeg', '.png')): continue

        stem = os.path.splitext(img_fname)[0]
        if stem in existing_ids: continue # 중복 방지

        # 추가 데이터의 JSON 경로 생성 (YOLO 로직과 동일)
        kid = stem.split('_')[0]
        json_path = os.path.join(EXTRA_JSON_DIR, kid + "_json", stem + ".json")

        if os.path.exists(json_path):
            with open(json_path, "r", encoding="utf-8") as f:
                extra_data = json.load(f)

            for ann in extra_data["annotations"]:
                x, y, w, h = ann["bbox"]
                extra_records.append({
                    "image_path": os.path.join(EXTRA_IMG_DIR, img_fname),
                    "image_id": stem,
                    "category_id": int(ann["category_id"]),
                    "bbox_x": float(x), "bbox_y": float(y), "bbox_w": float(w), "bbox_h": float(h),
                })

    extra_df = pd.DataFrame(extra_records)
    df = pd.concat([df, extra_df], ignore_index=True)
    print(f"✅ 추가 데이터 {len(extra_records)}개 병합 완료. 총 객체 수: {len(df)}")
else:
    print("⚠️ 추가 이미지 경로를 찾을 수 없습니다. 기본 데이터만 사용합니다.")

✅ 추가 데이터 982개 병합 완료. 총 객체 수: 5508


In [8]:
############################################################
# 2. category_id 매핑 (겉으로는 안 바꾸고, 모델 내부에서만 사용)
############################################################

# 원본 category_id 집합
unique_cats = sorted(df["category_id"].unique())
print("고유 category_id 개수:", len(unique_cats))

# 내부용: 모델에 넣을 label (1 ~ num_classes-1), 0은 background
orig2model = {cid: i + 1 for i, cid in enumerate(unique_cats)}   # 원본 → 모델용
model2orig = {v: k for k, v in orig2model.items()}               # 모델용 → 원본

num_classes = len(unique_cats) + 1  # background 포함
print("num_classes (background 포함):", num_classes)

고유 category_id 개수: 74
num_classes (background 포함): 75


In [9]:
############################################################
# 3. Dataset 정의
############################################################

class OralDrugDataset(Dataset):
    """
    Faster R-CNN용 Dataset
    - __getitem__은 (image, target) 반환
    - image: [C,H,W] float tensor
    - target: dict(boxes, labels, image_id, ...)
    """
    def __init__(self, df, orig2model, transforms=None):
        self.df = df.reset_index(drop=True)
        self.orig2model = orig2model
        self.transforms = transforms

        # 이미지 단위로 그룹을 만들기 위해 고유 image_id 리스트를 유지
        self.image_ids = self.df["image_id"].unique().tolist()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        # 1) image_id 선택
        image_id = self.image_ids[idx]

        # 2) 해당 image_id에 해당하는 모든 row (여러 객체) 가져오기
        df_img = self.df[self.df["image_id"] == image_id]

        # 3) 이미지 로드
        img_path = df_img["image_path"].iloc[0]
        image = Image.open(img_path).convert("RGB")

        # 4) bbox / labels 구성
        boxes = []
        labels = []

        for _, row in df_img.iterrows():
            x = row["bbox_x"]
            y = row["bbox_y"]
            w = row["bbox_w"]
            h = row["bbox_h"]

            # [x1, y1, x2, y2] 형식으로 변환
            x1 = x
            y1 = y
            x2 = x + w
            y2 = y + h
            boxes.append([x1, y1, x2, y2])

            # 원본 category_id → 모델용 label로 변환
            orig_cat = int(row["category_id"])
            model_cat = self.orig2model[orig_cat]
            labels.append(model_cat)

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            # image_id는 loss에는 크게 영향 없음, 그냥 인덱스 정도로 넣어도 무방
            "image_id": torch.tensor([idx]),
        }

        if self.transforms is not None:
            image = self.transforms(image)

        return image, target


############################################################
# 4. Transform, Dataset, DataLoader 구성
############################################################

train_transforms = T.Compose([
    # 필요하다면 여기서 RandomHorizontalFlip 등 augmentation 추가
    T.ToTensor(),  # PIL 이미지를 [0,1] float tensor로 변환
])

dataset = OralDrugDataset(df, orig2model=orig2model, transforms=train_transforms)

# 간단히 train/val split (예: 90% train, 10% val)
indices = torch.randperm(len(dataset)).tolist()
split = int(0.9 * len(indices))
train_indices = indices[:split]
val_indices   = indices[split:]

train_dataset = torch.utils.data.Subset(dataset, train_indices)
val_dataset   = torch.utils.data.Subset(dataset, val_indices)

def collate_fn(batch):
    # detection 모델용 collate_fn: 리스트의 튜플을 튜플의 리스트로
    return tuple(zip(*batch))

train_loader = DataLoader(
    train_dataset,
    batch_size=2,              # GPU 메모리에 맞게 조정
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn,
)

print("train steps:", len(train_loader), "val steps:", len(val_loader))



train steps: 1112 val steps: 124


In [ ]:
############################################################
# [추가] EfficientNet-B3 backbone + FPN 정의
############################################################

class EfficientNetB3BackboneWithFPN(nn.Module):
    def __init__(self, pretrained=True, out_channels=256):
        super().__init__()

        weights = EfficientNet_B3_Weights.DEFAULT if pretrained else None
        base_model = efficientnet_b3(weights=weights)

        # EfficientNet-B3에서 중간 feature map 추출
        # 보통 stride 4, 8, 16, 32 수준의 feature를 사용
        return_nodes = {
            "features.2": "0",
            "features.3": "1",
            "features.5": "2",
            "features.7": "3",
        }

        self.body = create_feature_extractor(base_model, return_nodes=return_nodes)

        # 각 feature map의 channel 수 자동 추론
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 256, 256)
            feats = self.body(dummy)
            in_channels_list = [feat.shape[1] for feat in feats.values()]

        self.fpn = FeaturePyramidNetwork(
            in_channels_list=in_channels_list,
            out_channels=out_channels,
            extra_blocks=LastLevelMaxPool()
        )

        # FasterRCNN에서 필요
        self.out_channels = out_channels

    def forward(self, x):
        feats = self.body(x)   # dict 형태
        feats = OrderedDict((k, v) for k, v in feats.items())
        feats = self.fpn(feats)
        return feats


def get_fasterrcnn_efficientnet_b3(num_classes):
    backbone = EfficientNetB3BackboneWithFPN(pretrained=True, out_channels=256)

    # 일반적인 anchor 설정
    # 만약 객체가 작으면 (16, 32, 64, 128, 256)로 바꿔도 됨
    anchor_generator = AnchorGenerator(
        sizes=((32,), (64,), (128,), (256,), (512,)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 5
    )

    # FPN feature map 중 어떤 레벨을 ROI pooling에 쓸지 지정
    box_roi_pool = MultiScaleRoIAlign(
        featmap_names=["0", "1", "2", "3"],
        output_size=7,
        sampling_ratio=2
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=box_roi_pool,
        min_size=512,
        max_size=1024
    )

    return model


In [ ]:
############################################################
# 4.5 옵튜나 학습 (mAP 기반 최적화 - 30% Subset)
############################################################

import optuna
import torch.optim as optim
from torch.utils.data import Subset
import numpy as np
from tqdm.auto import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import json

# 0. 결과 일관성을 위한 시드 고정
np.random.seed(42)

# 0-1. 옵튜나 전용 데이터 로더 (30% Subset)
indices = np.arange(len(train_dataset))
np.random.shuffle(indices)
subset_size = int(0.3 * len(train_dataset))
optuna_subset_indices = indices[:subset_size]
optuna_train_subset = Subset(train_dataset, optuna_subset_indices)

optuna_train_loader = DataLoader(
    optuna_train_subset,
    batch_size=train_loader.batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

def set_bn_eval(m):
    if isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm)):
        m.eval()

# [mAP 최적화] mAP 계산을 위한 보조 함수
def evaluate_map(model, data_loader, device):
    model.eval()
    results = []
    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            for i, output in enumerate(outputs):
                image_id = targets[i]["image_id"].item()
                boxes = output["boxes"].cpu().numpy()
                scores = output["score"].cpu().numpy() if "score" in output else output["scores"].cpu().numpy()
                labels = output["labels"].cpu().numpy()

                for box, score, label in zip(boxes, scores, labels):
                    # COCO format: [x, y, w, h]
                    results.append({
                        "image_id": image_id,
                        "category_id": int(label),
                        "bbox": [float(box[0]), float(box[1]), float(box[2]-box[0]), float(box[3]-box[1])],
                        "score": float(score)
                    })

    if not results: return 0.0

    # 임시 JSON 저장 및 평가 (TEST_JSON_PATH는 기존 정의된 변수 사용)
    with open("temp_optuna.json", "w") as f:
        json.dump(results, f)

    try:
        coco_gt = COCO(TEST_JSON_PATH) # 실제 검증용 JSON 경로
        coco_dt = coco_gt.loadRes("temp_optuna.json")
        coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
        coco_eval.evaluate()
        coco_eval.accumulate()
        # [mAP 최적화] mAP @ IoU=0.50:0.95 점수 추출
        coco_eval.summarize()
        return coco_eval.stats[0]
    except:
        return 0.0

def objective(trial):
    # 1. 하이퍼파라미터 탐색 범위 (안정적 구간)
    lr_backbone = trial.suggest_float("lr_backbone", 1e-5, 1.5e-4, log=True)
    lr_head = trial.suggest_float("lr_head", 8e-5, 8e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 5e-4, log=True)

    # 2. 모델 초기화
    current_model = get_fasterrcnn_efficientnet_b3(num_classes)
    current_model.to(DEVICE)

    backbone_params = [p for n, p in current_model.named_parameters() if p.requires_grad and n.startswith("backbone")]
    head_params = [p for n, p in current_model.named_parameters() if p.requires_grad and not n.startswith("backbone")]

    optimizer = optim.AdamW([
        {"params": backbone_params, "lr": lr_backbone},
        {"params": head_params, "lr": lr_head},
    ], weight_decay=weight_decay)

    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == "cuda"))

    # 3. 학습 실행 (7 에포크)
    search_epochs = 7
    for epoch in range(search_epochs):
        current_model.train()
        current_model.backbone.apply(set_bn_eval)

        for images, targets in optuna_train_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(DEVICE.type == "cuda")):
                loss_dict = current_model(images, targets)
                losses = sum(loss for loss in loss_dict.values())

            scaler.scale(losses).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(current_model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()

    # [mAP 최적화] 최종 리턴값을 검증 세트의 mAP로 설정
    current_mAP = evaluate_map(current_model, val_loader, DEVICE)

    # 가망 없는 시도는 가지치기 (Pruning)를 위해 보고
    trial.report(current_mAP, search_epochs)

    return current_mAP

# 4. 최적화 실행부
print(f"🚀 mAP 기반 하이퍼파라미터 최적화 시작... (Subset {subset_size}장 사용)")

# [mAP 최적화] mAP는 높을수록 좋으므로 direction="maximize"
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=12)

print("\n✨ 최적화 완료!")
print(f"🥇 최고 mAP Score: {study.best_value:.4f}")
print("Best Parameters:", study.best_params)

NameError: name 'train_dataset' is not defined

In [ ]:
############################################################
# 5. 모델 정의 (Faster R-CNN + EfficientNet-B3 전이학습)
############################################################

# [수정] 옵튜나가 찾은 최적의 파라미터를 변수에 할당 (찾지 못했을 경우를 대비해 기본값 설정)
best_params = getattr(study, 'best_params', {
    'lr_backbone': 1e-4,
    'lr_head': 5e-4,
    'weight_decay': 1e-4
})

# 사전학습된 모델 로드
model = get_fasterrcnn_efficientnet_b3(num_classes)
model.to(DEVICE)

############################################################
# 6. 학습 루프 (EfficientNet-B3용 권장 버전)
############################################################

num_epochs = 15  # 기존 5보다 늘리는 걸 추천

# backbone / head 파라미터 분리 (도영님 로직 반영)
backbone_params = []
head_params = []

for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if name.startswith("backbone"):
        backbone_params.append(p)
    else:
        head_params.append(p)

# [수정] 옵티마이저 설정 시 옵튜나에서 찾은 최적의 lr과 weight_decay 적용
optimizer = optim.AdamW(
    [
        {"params": backbone_params, "lr": best_params['lr_backbone']}, # 옵튜나 값 자동 입력
        {"params": head_params, "lr": best_params['lr_head']},         # 옵튜나 값 자동 입력
    ],
    weight_decay=best_params['weight_decay'] # 옵튜나 값 자동 입력
)

# 스케줄러 설정
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# [수정] AMP (torch.amp.GradScaler 사용)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == "cuda"))

# 작은 batch에서 EfficientNet의 BN 흔들림 방지 함수
def set_bn_eval(module):
    if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d, nn.SyncBatchNorm)):
        module.eval()

best_val_loss = float("inf")

for epoch in range(num_epochs):
    ########################################################
    # 1) Train
    ########################################################
    model.train()

    # backbone 안의 BatchNorm은 고정 (전이학습 성능 유지용)
    model.backbone.apply(set_bn_eval)

    train_loss_sum = 0.0

    for images, targets in train_loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        optimizer.zero_grad(set_to_none=True)

        # [수정] 최신 autocast 문법 적용
        with torch.amp.autocast('cuda', enabled=(DEVICE.type == "cuda")):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

        scaler.scale(losses).backward()

        # gradient exploding 방지용
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        scaler.step(optimizer)
        scaler.update()

        train_loss_sum += losses.item()

    avg_train_loss = train_loss_sum / max(1, len(train_loader))

    ########################################################
    # 2) Validation loss
    ########################################################
    # torchvision detection 모델은 loss를 얻으려면 train 모드로 호출해야 함
    model.train()
    model.backbone.apply(set_bn_eval)

    val_loss_sum = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            with torch.amp.autocast('cuda', enabled=(DEVICE.type == "cuda")):
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())

            val_loss_sum += losses.item()

    avg_val_loss = val_loss_sum / max(1, len(val_loader))

    print(
        f"[Epoch {epoch+1}/{num_epochs}] "
        f"train_loss: {avg_train_loss:.4f} | "
        f"val_loss: {avg_val_loss:.4f}"
    )

    scheduler.step()

    # best 모델 저장
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_fasterrcnn_efficientnet_b3.pth")
        print("✅ best 모델 저장 완료")

# 마지막 모델도 저장
torch.save(model.state_dict(), "last_fasterrcnn_efficientnet_b3.pth")
print("✅ 최종 모델 저장 완료")

[Epoch 1/15] train_loss: 0.6351 | val_loss: 0.3265
✅ best 모델 저장 완료
[Epoch 2/15] train_loss: 0.3615 | val_loss: 0.1882
✅ best 모델 저장 완료
[Epoch 3/15] train_loss: 0.3181 | val_loss: 0.1340
✅ best 모델 저장 완료
[Epoch 4/15] train_loss: 0.2400 | val_loss: 0.2004
[Epoch 5/15] train_loss: 0.2453 | val_loss: 0.1451
[Epoch 6/15] train_loss: 0.1715 | val_loss: 0.0819
✅ best 모델 저장 완료
[Epoch 7/15] train_loss: 0.1525 | val_loss: 0.0813
✅ best 모델 저장 완료
[Epoch 8/15] train_loss: 0.1589 | val_loss: 0.0727
✅ best 모델 저장 완료
[Epoch 9/15] train_loss: 0.1424 | val_loss: 0.0709
✅ best 모델 저장 완료
[Epoch 10/15] train_loss: 0.1420 | val_loss: 0.0749
[Epoch 11/15] train_loss: 0.1400 | val_loss: 0.0730
[Epoch 12/15] train_loss: 0.1428 | val_loss: 0.0718
[Epoch 13/15] train_loss: 0.1383 | val_loss: 0.0715
[Epoch 14/15] train_loss: 0.1389 | val_loss: 0.0711
[Epoch 15/15] train_loss: 0.1375 | val_loss: 0.0678
✅ best 모델 저장 완료
✅ 최종 모델 저장 완료


In [ ]:
############################################################
# 7. test_images에 대해 예측 → submission.csv 생성
############################################################

# test 이미지 파일 목록 가져오기
test_files = [f for f in os.listdir(TEST_IMG_DIR) if f.endswith(".png")]
test_files = sorted(test_files)

model.eval()

rows = []
annotation_id = 1      # submission용 annotation_id 시작
score_threshold = 0.3  # 너무 낮은 점수는 제거 (필요에 따라 조정)

with torch.no_grad():
    for f in test_files:
        img_path = os.path.join(TEST_IMG_DIR, f)
        image = Image.open(img_path).convert("RGB")

        # image_id = 파일명에서 확장자 제거한 문자열 그대로 사용
        image_id = os.path.splitext(f)[0]

        img_tensor = T.ToTensor()(image).to(DEVICE)
        outputs = model([img_tensor])[0]

        keep = outputs["scores"].cpu() >= score_threshold
        boxes  = outputs["boxes"].cpu()[keep]
        labels = outputs["labels"].cpu()[keep]
        scores = outputs["scores"].cpu()[keep]

        for box, lab, sc in zip(boxes, labels, scores):
            x1, y1, x2, y2 = box.tolist()
            w = x2 - x1
            h = y2 - y1

            orig_cat = model2orig[int(lab.item())]

            rows.append({
                "annotation_id": annotation_id,
                "image_id": image_id,  # 문자열 그대로 사용
                "category_id": orig_cat + 1,
                "bbox_x": x1,
                "bbox_y": y1,
                "bbox_w": w,
                "bbox_h": h,
                "score": float(sc.item()),
            })

            annotation_id += 1

# DataFrame으로 만들고 저장
df_sub = pd.DataFrame(rows, columns=[
    "image_id", "category_id",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"
])
# 이미지 ID별로 점수 높은 순 정렬 후 상위 4개 추출
df_sub = df_sub.sort_values(by=["image_id", "score"], ascending=[True, False])
df_sub = df_sub.groupby("image_id").head(4)

# 3) 최종 annotation_id 부여 (1부터 순차적으로)
df_sub.insert(0, "annotation_id", range(1, len(df_sub) + 1))

# 4) CSV 저장
output_path = "final_submission.csv"
df_sub.to_csv(os.path.join(extract_path, output_path), index=False)

print(f"✅ 생성 완료: {output_path}")
print(f"📊 총 예측 객체 수: {len(df_sub)}")
print(df_sub.head())

✅ 생성 완료: final_submission.csv
📊 총 예측 객체 수: 3238
   annotation_id image_id  category_id      bbox_x      bbox_y      bbox_w  \
0              1        1        16551  554.889221   77.180023  406.072205   
1              2        1        27926  597.539856  663.536560  262.163696   
2              3        1        24850  174.561646  730.737732  176.191864   
3              4        1         1900  158.002106  251.699600  204.473938   
4              5       10        21771  645.472046  290.892303  187.733398   

       bbox_h     score  
0  400.284912  0.988307  
1  488.245422  0.985432  
2  306.837097  0.984372  
3  125.135056  0.980110  
4  185.313080  0.989625  


In [ ]:
############################################################
# 8. 모델 성능 평가 (mAP 측정)
############################################################

import json
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# 1. df_sub(Pandas)를 COCO 평가용 리스트로 변환
eval_results = []
for _, row in df_sub.iterrows():
    eval_results.append({
        "image_id": int(row["image_id"]),
        "category_id": int(row["category_id"]) -1, ############# cat ID 바로잡기
        "bbox": [row["bbox_x"], row["bbox_y"], row["bbox_w"], row["bbox_h"]],
        "score": float(row["score"])
    })

# 2. 임시 JSON 파일로 저장
temp_json = "temp_eval.json"
with open(temp_json, "w") as f:
    json.dump(eval_results, f)

# 3. COCO 평가 실행
coco_gt = COCO(TEST_JSON_PATH)
coco_dt = coco_gt.loadRes(temp_json)

coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

loading annotations into memory...
Done (t=0.32s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=1.30s).
Accumulating evaluation results...
DONE (t=0.81s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.368
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.395
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.394
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.368
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.947
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.947
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe


결론부터 말씀드리면, 현재 모델은 **"정답은 기가 막히게 잘 찾아내지만(Recall), 위치를 정밀하게 그리는 능력(Precision)은 아직 보완이 필요하다"** 고 요약할 수 있습니다.

<br>

### **📊 핵심 지표 분석 (Performance Review)**

#### **1. Precision vs Recall의 불균형**

* **mAP (IoU 0.50:0.95):** **0.374**
* **mAR (maxDets 100):** **0.898**

* **해석:** 리콜(Recall)이 **0.898**로 매우 높습니다. 이는 모델이 이미지 속 물체를 **거의 놓치지 않고 다 찾아내고 있다**는 뜻입니다. 반면 AP는 그에 비해 낮습니다. 즉, 물체를 찾긴 찾는데 **Bounding Box의 위치가 약간 어긋나 있거나, 오탐(False Positive)이 섞여 있을 확률**이 높습니다.

<br>

#### **2. IoU 임계값(Threshold)에 따른 변화**

* **AP @ 0.50:** **0.397**
* **AP @ 0.75:** **0.395**

    * **해석:** 보통 IoU 기준이 엄격해질수록(0.5 $\rightarrow$ 0.75) 점수가 팍 떨어지기 마련인데, 이 모델은 거의 차이가 없습니다.
    * **진단:** 이는 모델이 **Box를 아주 대충 그리거나, 아주 정확하게 그리거나 둘 중 하나**라는 뜻입니다. 어중간하게 틀리는 게 아니라 위치 정확도 면에서 어떤 일관된 특징(예: 항상 실제보다 조금 크게 그림)이 있을 수 있습니다.

<br>

#### **3. Object Size에 따른 성능 (중요!)**

* **Small / Medium:** **-1.000**
* **Large:** **0.374**

    * **해석:** `-1.000`은 해당 데이터가 **평가셋에 아예 존재하지 않는다**는 뜻입니다.
    * **진단:** 현재 재찬 님이 다루시는 데이터셋의 물체들은 모두 **'Large'** 카테고리에 속합니다. 작은 물체를 탐지할 걱정은 안 해도 되지만, 큰 물체의 전체적인 윤곽을 더 정밀하게 잡는 데 집중해야 합니다.

<br>

### **💡 1타 강사의 '성능 향상' 클리닉**

재찬 님, 점수를 더 올리고 싶다면 수강생들에게 다음 **3가지 튜닝 전략**을 가르쳐주며 적용해 보세요.

1. **Localization Loss 강화:** 현재 리콜은 충분하니, 박스 위치를 더 정확히 잡아야 합니다. `CIoU`나 `DIoU` 같은 Loss 함수를 쓰고 있는지 확인하고, Box Regression의 가중치를 조금 높여보세요.

2. **Confidence Threshold 튜닝:** 리콜이 0.9에 가깝다는 건 모델이 아주 자신 있게(혹은 너무 남발해서) 박스를 치고 있다는 겁니다. Confidence 임계값을 살짝 높여서 확실한 것만 남기면 AP(정밀도)가 올라갈 수 있습니다.

3. **데이터 증강(Augmentation) 점검:** 물체가 크기 때문에(Large), 이미지 가장자리에 걸린 물체를 잘 잡는지 확인해 보세요. `Random Crop` 보다는 `Scaling`이나 `Translation` 위주의 증강이 더 효과적일 수 있습니다.

<br>

### **🎯 최종 리포트**

> **"모델이 눈은 좋은데(Recall), 손재주가 살짝 부족합니다(Precision). 박스를 더 예쁘게 그리도록 가르치면 점수는 수직 상승할 겁니다!"**